In [7]:
#bert模型：构建并训练了一个简化版本的BERT架构模型，其主体结构包含了标准的嵌入层、6个Transformer模块（每个模块由多头自注意力机制和前馈网络组成，并辅以残差连接与层归一化）、以及一个用于分类的输出层。在数据处理流程上，代码加载了预先处理并保存为news_data.pkl的文件，其中文本已被转换为基于自定义词汇表的token索引序列；随后，代码对这些序列进行清洗、填充至统一长度，并将其划分为训练集、验证集和测试集。模型采用AdamW优化器和交叉熵损失函数进行了为期10个轮次的训练。

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import sys
import pickle
import time
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# 设置随机种子
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

class SimpleBERTClassifier(nn.Module):
    def __init__(self, vocab_size=30522, embedding_dim=128, hidden_dim=256, 
                 num_classes=2, dropout=0.1):
        super(SimpleBERTClassifier, self).__init__()
        
        # 嵌入层
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # 简化Transformer: 使用多个自注意力层
        self.attention_layers = nn.ModuleList([
            nn.MultiheadAttention(embedding_dim, num_heads=8, batch_first=True, dropout=dropout)
            for _ in range(6)
        ])
        
        # 前馈网络
        self.feed_forward_layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(embedding_dim, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, embedding_dim)
            )
            for _ in range(6)
        ])
        
        # Layer normalization
        self.layer_norms1 = nn.ModuleList([
            nn.LayerNorm(embedding_dim) for _ in range(6)
        ])
        self.layer_norms2 = nn.ModuleList([
            nn.LayerNorm(embedding_dim) for _ in range(6)
        ])
        
        # 分类头
        self.dropout = nn.Dropout(dropout)
        self.pooler = nn.Linear(embedding_dim, embedding_dim)
        self.classifier = nn.Linear(embedding_dim, num_classes)
        
    def forward(self, input_ids):
        batch_size, seq_len = input_ids.size()
        
        # 嵌入
        x = self.embedding(input_ids)
        
        # 创建注意力掩码
        attention_mask = (input_ids != 0)
        
        # 简化Transformer编码
        for i in range(6):
            # 残差连接1
            residual = x
            
            # 自注意力
            attn_output, _ = self.attention_layers[i](
                x, x, x, 
                key_padding_mask=~attention_mask
            )
            x = self.layer_norms1[i](x + attn_output)
            
            # 残差连接2
            residual2 = x
            
            # 前馈网络
            ff_output = self.feed_forward_layers[i](x)
            x = self.layer_norms2[i](x + ff_output)
        
        # 池化: 平均池化
        pooled_output = x.mean(dim=1)
        pooled_output = self.pooler(pooled_output)
        pooled_output = torch.tanh(pooled_output)
        pooled_output = self.dropout(pooled_output)
        
        # 分类
        logits = self.classifier(pooled_output)
        
        return logits

def load_saved_data(file_path):
    """加载保存的数据"""
    with open(file_path, 'rb') as f:
        data = pickle.load(f)
    return data

def prepare_data():
    """准备数据"""
    print("=" * 50)
    print("数据预处理")
    print("=" * 50)
    
    print("正在加载数据...")
    try:
        data = load_saved_data("news_data.pkl")
        X_train = data["X_train"]
        X_test = data["X_test"]
        y_train = data["y_train"]
        y_test = data["y_test"]
        word_to_idx = data["word_to_idx"]
        vocab_size = data["vocab_size"]
        
        print(f"数据加载成功！")
        print(f"训练集大小: {len(X_train)}")
        print(f"测试集大小: {len(X_test)}")
        print(f"词汇表大小: {vocab_size}")
    except Exception as e:
        print(f"加载数据失败: {e}")
        print("请确保 news_data.pkl 文件存在")
        sys.exit(1)
    
    # 检查并清洗数据
    def clean_sequence(seq):
        """清洗序列，确保所有元素都是整数"""
        cleaned = []
        for token in seq:
            if isinstance(token, (int, np.integer)):
                cleaned.append(int(token))
            elif isinstance(token, str):
                try:
                    cleaned.append(int(token))
                except:
                    cleaned.append(1)  # 使用<UNK>标记
            else:
                cleaned.append(1)
        return cleaned
    
    # 处理训练集和测试集
    print("正在清洗数据...")
    X_train_clean = [clean_sequence(seq) for seq in X_train]
    X_test_clean = [clean_sequence(seq) for seq in X_test]
    
    # 确保没有空序列
    X_train_clean = [seq for seq in X_train_clean if len(seq) > 0]
    X_test_clean = [seq for seq in X_test_clean if len(seq) > 0]
    
    print(f"清洗后训练集大小: {len(X_train_clean)}")
    print(f"清洗后测试集大小: {len(X_test_clean)}")
    
    # 分析序列长度
    train_lengths = [len(seq) for seq in X_train_clean]
    max_len = 128  # 固定长度，简化处理
    
    print(f"序列长度统计:")
    print(f"  训练集平均长度: {np.mean(train_lengths):.1f}")
    print(f"  训练集最大长度: {max(train_lengths)}")
    print(f"  选择最大长度: {max_len}")
    
    # 序列填充函数
    def pad_sequence(seq, max_len, padding_value=0):
        if len(seq) >= max_len:
            return seq[:max_len]
        else:
            return seq + [padding_value] * (max_len - len(seq))
    
    # 填充序列
    X_train_pad = [pad_sequence(seq, max_len) for seq in X_train_clean]
    X_test_pad = [pad_sequence(seq, max_len) for seq in X_test_clean]
    
    # 转换为PyTorch张量
    X_train_tensor = torch.tensor(X_train_pad, dtype=torch.long)
    y_train_tensor = torch.tensor(y_train[:len(X_train_pad)], dtype=torch.long)
    X_test_tensor = torch.tensor(X_test_pad, dtype=torch.long)
    y_test_tensor = torch.tensor(y_test[:len(X_test_pad)], dtype=torch.long)
    
    # 划分验证集
    train_size = int(0.8 * len(X_train_tensor))
    val_size = len(X_train_tensor) - train_size
    
    if train_size > 0 and val_size > 0:
        X_train_final, X_val_final = X_train_tensor.split([train_size, val_size])
        y_train_final, y_val_final = y_train_tensor.split([train_size, val_size])
    else:
        # 如果数据集太小，使用全部数据
        X_train_final, X_val_final = X_train_tensor, X_train_tensor[:0]
        y_train_final, y_val_final = y_train_tensor, y_train_tensor[:0]
    
    print(f"\n数据集划分:")
    print(f"  训练集: {len(X_train_final)}")
    print(f"  验证集: {len(X_val_final)}")
    print(f"  测试集: {len(X_test_tensor)}")
    
    return (X_train_final, y_train_final, 
            X_val_final, y_val_final, 
            X_test_tensor, y_test_tensor, 
            max_len, vocab_size)

def train_epoch(model, data_loader, criterion, optimizer, device):
    """训练一个epoch"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for data, target in data_loader:
        data, target = data.to(device), target.to(device)
        
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(output.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()
    
    avg_loss = total_loss / len(data_loader) if len(data_loader) > 0 else 0
    accuracy = correct / total if total > 0 else 0
    
    return avg_loss, accuracy

def evaluate_model(model, data_loader, criterion, device):
    """评估模型"""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for data, target in data_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)
            
            total_loss += loss.item()
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
    
    avg_loss = total_loss / len(data_loader) if len(data_loader) > 0 else 0
    accuracy = correct / total if total > 0 else 0
    
    return avg_loss, accuracy

def train_model():
    """训练模型"""
    print("=" * 50)
    print("BERT模型训练")
    print("=" * 50)
    
    # 1. 准备数据
    (X_train, y_train, 
     X_val, y_val, 
     X_test, y_test, 
     max_len, vocab_size) = prepare_data()
    
    # 2. 创建DataLoader
    batch_size = 16
    train_dataset = TensorDataset(X_train, y_train)
    val_dataset = TensorDataset(X_val, y_val) if len(X_val) > 0 else None
    test_dataset = TensorDataset(X_test, y_test)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False) if val_dataset else None
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    # 3. 初始化模型
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"使用设备: {device}")
    
    # 模型配置
    config = {
        'vocab_size': min(vocab_size, 30000),
        'embedding_dim': 128,
        'hidden_dim': 256,
        'num_classes': 2,
        'dropout': 0.1
    }
    
    model = SimpleBERTClassifier(**config).to(device)
    
    # 4. 计算参数量
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"模型总参数量: {total_params:,}")
    print(f"可训练参数量: {trainable_params:,}")
    print(f"模型配置: {config}")
    
    # 5. 损失函数和优化器
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-4,
        weight_decay=0.01
    )
    
    # 6. 训练循环
    num_epochs = 10
    best_val_acc = 0
    training_history = []
    
    print(f"\n开始训练，共 {num_epochs} 个epoch")
    
    for epoch in range(num_epochs):
        start_time = time.time()
        
        # 训练
        train_loss, train_acc = train_epoch(
            model, train_loader, criterion, optimizer, device
        )
        
        # 验证
        if val_loader:
            val_loss, val_acc = evaluate_model(
                model, val_loader, criterion, device
            )
        else:
            val_loss, val_acc = 0, train_acc
        
        epoch_time = time.time() - start_time
        
        # 记录历史
        history_entry = {
            'epoch': epoch + 1,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'val_loss': val_loss,
            'val_acc': val_acc,
            'time': epoch_time
        }
        training_history.append(history_entry)
        
        # 输出信息
        print(f"Epoch {epoch+1:2d}/{num_epochs} | "
              f"时间: {epoch_time:.1f}s | "
              f"训练 Loss: {train_loss:.4f} | 训练 Acc: {train_acc:.4f} | ", end="")
        
        if val_loader:
            print(f"验证 Loss: {val_loss:.4f} | 验证 Acc: {val_acc:.4f}")
        else:
            print("无验证集")
        
        # 保存最佳模型
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'config': config
            }, 'best_bert_model.pth')
            print(f"  ✓ 保存最佳模型 (验证准确率: {val_acc:.4f})")
    
    # 7. 测试模型
    print("\n" + "=" * 50)
    print("测试模型")
    print("=" * 50)
    
    # 加载最佳模型
    try:
        checkpoint = torch.load('best_bert_model.pth', map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
    except:
        print("使用当前模型进行测试")
    
    # 评估
    test_loss, test_acc = evaluate_model(
        model, test_loader, criterion, device
    )
    
    print(f"测试集结果:")
    print(f"  测试损失: {test_loss:.4f}")
    print(f"  测试准确率: {test_acc:.4f}")
    
    target_acc = 0.7
    if test_acc >= target_acc:
        print(f"  ✓ 达到目标，测试准确率 ({test_acc:.4f}) 高于目标值 {target_acc}")
    else:
        print(f"  ⚠ 未达目标，测试准确率 ({test_acc:.4f}) 低于目标值 {target_acc}")
        print("\n建议调整以下超参数:")
        print("  1. 增加训练轮数")
        print("  2. 调整学习率")
        print("  3. 增加模型维度")
        print("  4. 增加batch_size")
    
    # 8. 预测示例
    print("\n" + "=" * 50)
    print("预测示例")
    print("=" * 50)
    
    # 选择5个样本进行预测
    num_samples = min(5, len(X_test))
    model.eval()
    with torch.no_grad():
        for i in range(num_samples):
            sample = X_test[i].unsqueeze(0).to(device)
            target = y_test[i].item()
            
            output = model(sample)
            probs = F.softmax(output, dim=1)
            pred_prob, pred_class = torch.max(probs, 1)
            
            pred_prob_val = pred_prob.item()
            pred_class_val = pred_class.item()
            
            correct = "✓" if pred_class_val == target else "✗"
            
            print(f"样本 {i+1}:")
            print(f"  真实标签: {target} | 预测标签: {pred_class_val}")
            print(f"  预测概率: {pred_prob_val:.4f} | 结果: {correct}")
            if i < num_samples - 1:
                print()
    
    # 9. 保存完整模型
    torch.save({
        'model_state_dict': model.state_dict(),
        'config': config,
        'max_len': max_len,
        'vocab_size': vocab_size,
        'test_acc': test_acc,
        'history': training_history
    }, 'bert_classifier_complete.pth')
    
    print("\n保存完整模型信息...")
    print("模型已保存:")
    print("  - best_bert_model.pth (训练检查点)")
    print("  - bert_classifier_complete.pth (完整模型)")
    
    return {
        'model': model,
        'test_acc': test_acc,
        'config': config,
        'history': training_history
    }

def main():
    """主函数"""
    print("BERT模型 - 20newsgroups文本分类")
    print("=" * 50)
    
    # 训练模型
    results = train_model()
    
    print(f"\n训练完成！最终测试准确率: {results['test_acc']:.4f}")
    
    # 输出训练总结
    if results['history']:
        best_epoch = np.argmax([h['val_acc'] for h in results['history']]) + 1
        best_val_acc = max([h['val_acc'] for h in results['history']])
        print(f"最佳验证准确率: {best_val_acc:.4f} (Epoch {best_epoch})")
    
    return results

if __name__ == "__main__":
    main()

BERT模型 - 20newsgroups文本分类
BERT模型训练
数据预处理
正在加载数据...
数据加载成功！
训练集大小: 1079
测试集大小: 717
词汇表大小: 11753
正在清洗数据...
清洗后训练集大小: 1058
清洗后测试集大小: 695
序列长度统计:
  训练集平均长度: 1283.7
  训练集最大长度: 45731
  选择最大长度: 128

数据集划分:
  训练集: 846
  验证集: 212
  测试集: 695
使用设备: cpu
模型总参数量: 2,316,034
可训练参数量: 2,316,034
模型配置: {'vocab_size': 11753, 'embedding_dim': 128, 'hidden_dim': 256, 'num_classes': 2, 'dropout': 0.1}

开始训练，共 10 个epoch
Epoch  1/10 | 时间: 24.6s | 训练 Loss: 0.7026 | 训练 Acc: 0.5201 | 验证 Loss: 0.6939 | 验证 Acc: 0.5094
  ✓ 保存最佳模型 (验证准确率: 0.5094)
Epoch  2/10 | 时间: 24.9s | 训练 Loss: 0.7015 | 训练 Acc: 0.5307 | 验证 Loss: 0.6964 | 验证 Acc: 0.5047
Epoch  3/10 | 时间: 22.1s | 训练 Loss: 0.6961 | 训练 Acc: 0.5544 | 验证 Loss: 0.7598 | 验证 Acc: 0.5047
Epoch  4/10 | 时间: 22.7s | 训练 Loss: 0.6960 | 训练 Acc: 0.5508 | 验证 Loss: 0.7019 | 验证 Acc: 0.5094
Epoch  5/10 | 时间: 21.2s | 训练 Loss: 0.6887 | 训练 Acc: 0.5745 | 验证 Loss: 0.7114 | 验证 Acc: 0.5047
Epoch  6/10 | 时间: 21.5s | 训练 Loss: 0.6902 | 训练 Acc: 0.5591 | 验证 Loss: 0.7239 | 验证 Acc: 0.5047
Epoch  7/1